# 02 — Model Architecture

Inspect the MedicalVMamba architecture:
- Parameter count per component
- Forward pass shape tracing
- Feature map resolution at each stage

In [7]:
import torch
from medical_mamba.models import MedicalVMamba

## 1. Build the Model

In [8]:
model = MedicalVMamba(
    task_configs=[("pathmnist", 9)],
    backbone_cfg={
        "in_chans": 3,
        "depths": [2, 2, 9, 2],
        "embed_dim": 96,
        "drop_path_rate": 0.2,
        "expand": 2,
        "patch_size": 4,
    },
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params     : {total_params:,}")
print(f"Trainable params : {trainable_params:,}")


Total params     : 59,767,401
Trainable params : 59,767,401


## 2. Per-Component Breakdown

In [9]:
for name, module in model.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f"{name:20s} → {n:>12,} params")

backbone             →   59,298,528 params
heads                →        8,457 params
domain_projector     →      460,416 params


## 3. Forward Pass Shape Tracing

In [10]:
x = torch.randn(1, 3, 224, 224)
logits, intermediates = model(x, task_name="pathmnist")

print(f"Input  : {x.shape}")
for i, feat in enumerate(intermediates):
    print(f"Stage {i}: {feat.shape}")
print(f"Logits : {logits.shape}")


Input  : torch.Size([1, 3, 224, 224])
Stage 0: torch.Size([1, 3136, 96])
Stage 1: torch.Size([1, 784, 192])
Stage 2: torch.Size([1, 196, 384])
Stage 3: torch.Size([1, 49, 768])
Logits : torch.Size([1, 9])
